# Accessibility Inequality Analysis — Theil Decomposition

This notebook computes the population-weighted **Theil index**, which splits total
inequality into a *between-group* and a *within-group* component. The
decomposition is run along three dimensions: **municipality**, **socio-economic
status** (SES tercile), and **national origin** (national / other-EU / non-EU).



## Folder structure

All input files live in a `data/` folder next to this notebook. Run the
notebook from its own directory so the relative paths below resolve correctly.

```text
Phase 3/
├── compute_theil.ipynb                          # this notebook — Theil decomposition
├── create_grid_to_municipality.ipynb          # run this first to build grid_to_municipality.csv
└── data/
    ├── accessibility_results_sigma_20p0.csv   # Phase 2 accessibility scores 
    ├── origins.csv                            # population grid
    ├── lux_census_2021_socio.csv              # 2021 census nationality breakdown
    └── grid_to_municipality.csv               # produced by create_grid_to_municipality.ipynb
```


## Required columns per input file

| File | Required columns | Notes |
|------|------------------|-------|
| `accessibility_results_sigma_20p0.csv` | `GRD_ID`, and `A_<category>_A` + `A_<category>_D` for every category | |
| `origins.csv` | `GRD_ID`, `Pop_grids` | `Pop_grids` is the cell population; privacy-suppressed cells may hold `<30` (parsed to a midpoint of 15). |
| `lux_census_2021_socio.csv` | `SPATIAL`, `NAT`, `EU_OTH`, `OTH` | `SPATIAL` is joined to `GRD_ID`. `NAT` / `EU_OTH` / `OTH` are national / other-EU / non-EU population counts; NA or all-zero rows are imputed. |
| `grid_to_municipality.csv` | `GRD_ID`, `municipality`, `ses_tercile` | |


In [1]:
import pandas as pd
import numpy as np
import re


In [2]:
# ── CONFIG ──────────────────────────────────────────────────────────────

ACC_PATH     = "data/accessibility_results_sigma_20p0.csv"
ORIGINS_PATH = "data/origins.csv"               # Phase 1 population grid
CENSUS_PATH  = "data/lux_census_2021_socio.csv" # nationality breakdown
MUNI_PATH    = "data/grid_to_municipality.csv"  # GRD_ID, municipality, ses_tercile
SCENARIOS  = ["A", "D"]
CATEGORIES = [
    "entertainment", "jobs", "commercial", "bars_and_restaurants",
    "food", "education", "parks", "basic_healthcare"
]
EPS = 1e-9
SUPPRESSED_MIDPOINT = 15  # midpoint for <30 cells, consistent with Phase 1



In [3]:
# ── LOAD ACCESSIBILITY ──────────────────────────────────────────────────
acc = pd.read_csv(ACC_PATH)

print("done")
# ── MEAN-NORMALIZATION ACROSS SCENARIOS ─────────────────────────────────

for cat in CATEGORIES:
    raw_A = f"A_{cat}_A"
    denom = acc[raw_A].mean()
    if denom <= 0 or not np.isfinite(denom):
        for scen in SCENARIOS:
            acc[f"A_{cat}_{scen}_norm"] = 1.0
        continue
    for scen in SCENARIOS:
        raw_col = f"A_{cat}_{scen}"
        acc[f"{raw_col}_norm"] = acc[raw_col] / denom

# ── AGGREGATE NORMALIZED ACCESSIBILITY ─────────────────────────────────
for scen in SCENARIOS:
    norm_cols = [f"A_{cat}_{scen}_norm" for cat in CATEGORIES]
    acc[f"A_agg_{scen}"] = acc[norm_cols].mean(axis=1)

# ── LOAD ORIGINS  ──────────────────────
# Parse Pop_grids: exact numbers kept as-is; '<30' cells → midpoint of 15.
def parse_pop_value(x, midpoint=SUPPRESSED_MIDPOINT):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    try:
        return float(s)
    except ValueError:
        pass
    if re.match(r"^\s*<=?\s*30\s*$", s):
        return float(midpoint)
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        return float(m.group(1))
    return np.nan

origins = pd.read_csv(ORIGINS_PATH)
origins["T"] = origins["Pop_grids"].apply(parse_pop_value)
origins["is_suppressed"] = origins["Pop_grids"].apply(
    lambda x: bool(re.match(r"^\s*<=?\s*30\s*$", str(x).strip())) if isinstance(x, str) else False
)

n_suppressed = origins["is_suppressed"].sum()
print(f"Total cells in origins:       {len(origins)}")
print(f"Suppressed cells (<30):       {n_suppressed} ({n_suppressed/len(origins)*100:.1f}%)")
print(f"Total population (origins):   {origins['T'].sum():,.0f}")



done
Total cells in origins:       2795
Suppressed cells (<30):       567 (20.3%)
Total population (origins):   646,170


In [4]:
# ── LOAD CENSUS (nationality breakdown only) ─────────────────────────────
census = pd.read_csv(CENSUS_PATH)

# ── COMPUTE NATIONAL AVERAGE SHARES from non-suppressed, complete cells ──
# These shares are used to impute NAT/EU_OTH/OTH for suppressed cells
# where the census reports NAs or zero population.
census_complete = census[
    census["NAT"].notna() &
    census["EU_OTH"].notna() &
    census["OTH"].notna()
].copy()
census_complete["T_census"] = census_complete[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
census_complete = census_complete[census_complete["T_census"] > 0]

total_complete = census_complete["T_census"].sum()
nat_share   = census_complete["NAT"].sum()    / total_complete
eu_oth_share= census_complete["EU_OTH"].sum() / total_complete
oth_share   = census_complete["OTH"].sum()    / total_complete

print(f"\nNational average shares (from {len(census_complete)} complete cells):")
print(f"  NAT:    {nat_share:.3f}")
print(f"  EU_OTH: {eu_oth_share:.3f}")
print(f"  OTH:    {oth_share:.3f}")

# ── MERGE ORIGINS → CENSUS ───────────────────────────────────────────────
pop = origins[["GRD_ID", "T", "is_suppressed"]].merge(
    census[["SPATIAL", "NAT", "EU_OTH", "OTH"]],
    left_on="GRD_ID", right_on="SPATIAL", how="left"
)

# ── IMPUTE NAT/EU_OTH/OTH FOR PROBLEMATIC CELLS ─────────────────────────
# A cell needs imputation when:
#   (a) census NAT/EU_OTH/OTH are NA, OR
#   (b) census NAT+EU_OTH+OTH = 0  (census treats cell as uninhabited)

pop["T_census"] = pop[["NAT", "EU_OTH", "OTH"]].sum(axis=1)
needs_imputation = pop["NAT"].isna() | (pop["T_census"] == 0)

pop.loc[needs_imputation, "NAT"]    = pop.loc[needs_imputation, "T"] * nat_share
pop.loc[needs_imputation, "EU_OTH"] = pop.loc[needs_imputation, "T"] * eu_oth_share
pop.loc[needs_imputation, "OTH"]    = pop.loc[needs_imputation, "T"] * oth_share

n_imputed = needs_imputation.sum()
print(f"\nCells imputed (NA or zero in census): {n_imputed}")
print(f"Population in imputed cells:          {pop.loc[needs_imputation, 'T'].sum():,.0f}")
print(f"Total population after imputation: {pop['T'].sum():,.0f}")




National average shares (from 985 complete cells):
  NAT:    0.501
  EU_OTH: 0.341
  OTH:    0.158

Cells imputed (NA or zero in census): 1508
Population in imputed cells:          5,367
Total population after imputation: 646,170


In [5]:
# ── LOAD MUNICIPALITY ────────────────────────────────────────────────────
muni = pd.read_csv(MUNI_PATH)

# ── MERGE ALL ────────────────────────────────────────────────────────────
df = acc.merge(pop[["GRD_ID", "T", "NAT", "EU_OTH", "OTH"]], on="GRD_ID", how="inner")
df = df.merge(muni, on="GRD_ID", how="inner")

required_cols = ["GRD_ID", "T", "municipality", "ses_tercile", "NAT", "EU_OTH", "OTH"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after merge: {missing}")

print(f"\nCells in final df: {len(df)}")
print(f"A_agg_A: mean={df['A_agg_A'].mean():.4f}, "
      f"min={df['A_agg_A'].min():.4f}, max={df['A_agg_A'].max():.4f}")
print(f"A_agg_D: mean={df['A_agg_D'].mean():.4f}, "
      f"min={df['A_agg_D'].min():.4f}, max={df['A_agg_D'].max():.4f}")



Cells in final df: 2795
A_agg_A: mean=1.0000, min=0.0000, max=21.6208
A_agg_D: mean=1.0170, min=0.0000, max=22.8354


In [6]:
# ── THEIL FUNCTIONS ─────────────────────────────────────────────────────
def weighted_theil_total(data, access_col, weight_col):
    d = data[[access_col, weight_col]].dropna().copy()
    d = d[d[weight_col] > 0].copy()
    if len(d) == 0:
        return None
    d[access_col] = d[access_col].clip(lower=EPS)
    W = d[weight_col].sum()
    mu = (d[access_col] * d[weight_col]).sum() / W
    ratio = d[access_col] / mu
    T = (d[weight_col] * ratio * np.log(ratio)).sum() / W
    return {"T_total": T, "n_cells": len(d), "n_pop": W}


def theil_decompose(data, access_col, group_col, weight_col):
    d = data[[access_col, group_col, weight_col]].dropna().copy()
    d = d[d[weight_col] > 0].copy()
    if len(d) == 0:
        return None
    d[access_col] = d[access_col].clip(lower=EPS)

    W = d[weight_col].sum()
    mu = (d[access_col] * d[weight_col]).sum() / W
    ratio = d[access_col] / mu
    T_total = (d[weight_col] * ratio * np.log(ratio)).sum() / W

    T_between = 0.0
    T_within_direct = 0.0
    for g, sub in d.groupby(group_col):
        Wg = sub[weight_col].sum()
        if Wg <= 0:
            continue
        mu_g = (sub[access_col] * sub[weight_col]).sum() / Wg
        mu_g = max(mu_g, EPS)
        share_g = Wg / W
        r_g = mu_g / mu
        T_between += share_g * r_g * np.log(r_g)
        ratio_g = sub[access_col] / mu_g
        T_g = (sub[weight_col] * ratio_g * np.log(ratio_g)).sum() / Wg
        T_within_direct += share_g * r_g * T_g

    T_within = T_total - T_between
    return {
        "T_total": T_total,
        "T_between": T_between,
        "T_within": T_within,
        "T_within_direct": T_within_direct,
        "between_share_pct": (T_between / T_total * 100) if T_total > 0 else 0.0,
        "within_share_pct": (T_within / T_total * 100) if T_total > 0 else 0.0,
        "n_cells": len(d),
        "n_pop": W,
        "n_groups": d[group_col].nunique(),
    }


def theil_origin(data, access_col, origin_cols=("NAT", "EU_OTH", "OTH")):
    parts = []
    for g in origin_cols:
        sub = data[[access_col, g]].dropna().copy()
        sub = sub[sub[g] > 0].copy()
        if len(sub) == 0:
            continue
        sub["group"] = g
        sub["weight"] = sub[g]
        parts.append(sub[[access_col, "group", "weight"]])
    if not parts:
        return None
    long = pd.concat(parts, ignore_index=True)
    return theil_decompose(long, access_col, "group", "weight")


In [7]:
# ── THEIL ON ACCESSIBILITY LEVELS ─────────────────────────────────
results = []

for scen in SCENARIOS:
    ac = f"A_agg_{scen}"

    overall = weighted_theil_total(df, ac, "T")
    results.append({
        "scenario": scen, "decomposition": "overall",
        "T_total": overall["T_total"],
        "T_between": np.nan, "T_within": np.nan, "T_within_direct": np.nan,
        "between_share_pct": np.nan, "within_share_pct": np.nan,
        "n_cells": overall["n_cells"], "n_pop": overall["n_pop"],
        "n_groups": np.nan,
    })

    r = theil_decompose(df, ac, "municipality", "T")
    results.append({"scenario": scen, "decomposition": "municipality", **r})

    r = theil_decompose(df, ac, "ses_tercile", "T")
    results.append({"scenario": scen, "decomposition": "ses_tercile", **r})

    r = theil_origin(df, ac)
    results.append({"scenario": scen, "decomposition": "origin", **r})

res = pd.DataFrame(results)
mask = res["decomposition"] != "overall"
res.loc[mask, "decomp_check_absdiff"] = np.abs(
    res.loc[mask, "T_within"] - res.loc[mask, "T_within_direct"]
)

res.to_csv("theil_results.csv", index=False)

print(res[[
    "scenario", "decomposition", "T_total", "T_between",
    "T_within", "between_share_pct", "within_share_pct",
]].to_string(index=False))


scenario decomposition  T_total  T_between  T_within  between_share_pct  within_share_pct
       A       overall 0.440065        NaN       NaN                NaN               NaN
       A  municipality 0.440065   0.346900  0.093165          78.829289         21.170711
       A   ses_tercile 0.440065   0.102727  0.337338          23.343645         76.656355
       A        origin 0.439550   0.029526  0.410024           6.717338         93.282662
       D       overall 0.445768        NaN       NaN                NaN               NaN
       D  municipality 0.445768   0.352609  0.093159          79.101490         20.898510
       D   ses_tercile 0.445768   0.103830  0.341938          23.292426         76.707574
       D        origin 0.445251   0.029988  0.415263           6.735174         93.264826


In [8]:
# ── THEIL ON THE GAIN VECTOR  ─────────────────
df["gain"] = df["A_agg_D"] - df["A_agg_A"]

pop_total = df["T"].sum()
pop_zero  = df.loc[df["gain"] <= 0, "T"].sum()
print(f"Population with zero gain: {pop_zero:,.0f} / {pop_total:,.0f} "
      f"({pop_zero/pop_total*100:.1f}%)")
print(f"Cells with zero gain: {(df['gain'] <= 0).sum()} / {len(df)}")

df_pos = df[df["gain"] > 0].copy()

gain_results = []

overall = weighted_theil_total(df_pos, "gain", "T")
gain_results.append({
    "decomposition": "overall",
    "T_total": overall["T_total"],
    "T_between": np.nan, "T_within": np.nan, "T_within_direct": np.nan,
    "between_share_pct": np.nan, "within_share_pct": np.nan,
    "n_cells": overall["n_cells"], "n_pop": overall["n_pop"],
    "n_groups": np.nan,
})

r = theil_decompose(df_pos, "gain", "municipality", "T")
gain_results.append({"decomposition": "municipality", **r})

r = theil_decompose(df_pos, "gain", "ses_tercile", "T")
gain_results.append({"decomposition": "ses_tercile", **r})

r = theil_origin(df_pos, "gain")
gain_results.append({"decomposition": "origin", **r})

gain_res = pd.DataFrame(gain_results)
mask = gain_res["decomposition"] != "overall"
gain_res.loc[mask, "decomp_check_absdiff"] = np.abs(
    gain_res.loc[mask, "T_within"] - gain_res.loc[mask, "T_within_direct"]
)

gain_res.to_csv("theil_gain.csv", index=False)

print("\n" + gain_res[[
    "decomposition", "T_total", "T_between",
    "T_within", "between_share_pct", "within_share_pct",
    "n_cells", "n_groups",
]].to_string(index=False))


Population with zero gain: 43,922 / 646,170 (6.8%)
Cells with zero gain: 1189 / 2795

decomposition  T_total  T_between  T_within  between_share_pct  within_share_pct  n_cells  n_groups
      overall 1.043894        NaN       NaN                NaN               NaN     1170       NaN
 municipality 1.043894   0.629520  0.414375          60.304926         39.695074     1170     100.0
  ses_tercile 1.043894   0.145175  0.898720          13.907031         86.092969     1170       3.0
       origin 1.043582   0.046388  0.997193           4.445088         95.554912     3154       3.0


In [11]:
# ── WHERE ARE THE ZERO-GAIN CELLS? ──────────────────────────────────────
df_zero = df[df["gain"] <= 0].copy()

# by municipality
zero_by_muni  = df_zero.groupby("municipality")["T"].sum().sort_values(ascending=False)
total_by_muni = df.groupby("municipality")["T"].sum()
share_zero    = (zero_by_muni / total_by_muni).sort_values(ascending=False)

# by SES
zero_by_ses  = df_zero.groupby("ses_tercile")["T"].sum()
total_by_ses = df.groupby("ses_tercile")["T"].sum()
share_zero_ses = zero_by_ses / total_by_ses

# by origin
zero_origin  = df_zero[["NAT", "EU_OTH", "OTH"]].sum()
total_origin = df[["NAT", "EU_OTH", "OTH"]].sum()
share_zero_origin = zero_origin / total_origin

print(f"Share with zero gain by origin:\n{share_zero_origin}")


Share with zero gain by origin:
NAT       0.074440
EU_OTH    0.063628
OTH       0.054585
dtype: float64
